In [1]:
ord("😀")

128512

In [2]:
list("keurfyieugfieugf".encode("utf-8"))

[107,
 101,
 117,
 114,
 102,
 121,
 105,
 101,
 117,
 103,
 102,
 105,
 101,
 117,
 103,
 102]

In [3]:
text = """Contains the base Tokenizer class and a few common helper functions.
The base class also contains the (common) save/load functionality.
It would be possible to be a lot more strict about the interface and
e.g. isolating all regex/pattern parts to the RegexTokenizer, but
some concessions are made for simplicity."""
raw_tokens = text.encode("utf-8")
tokens = list(map(int,raw_tokens)) # first convert everything to integers and then make the whole thing a list 


In [4]:
with open("input.txt",'r',encoding="utf-8") as f:
    text = f.read()
tokens = list(text.encode('utf-8'))


In [5]:
def get_pairwise(ids):
    counts = {}
    for pair in zip(ids,ids[1:]):
        counts[pair] = counts.get(pair,0) + 1
    return counts
stats = get_pairwise(tokens)
#print(sorted(((v,k) for k,v in stats.items()),reverse= True))

In [6]:
top_pair = max(stats,key = stats.get)
def merge(ids,pair,idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids
new_stats = merge(tokens,top_pair,256)


In [7]:
vocab_size = 356
num_merges = vocab_size - 256
ids = list(tokens)
merges = {} # int,int -> int
for i in range(num_merges):
    stats = get_pairwise(ids)
    max_pair = max(stats, key = stats.get)
    idx = 256 + i
    ids = merge(ids,max_pair,idx)
    merges[max_pair] = idx
    print(f"We merged {max_pair} to be {idx}")


We merged (101, 32) to be 256
We merged (116, 104) to be 257
We merged (116, 32) to be 258
We merged (115, 32) to be 259
We merged (100, 32) to be 260
We merged (44, 32) to be 261
We merged (111, 117) to be 262
We merged (101, 114) to be 263
We merged (105, 110) to be 264
We merged (121, 32) to be 265
We merged (97, 110) to be 266
We merged (58, 10) to be 267
We merged (111, 114) to be 268
We merged (111, 32) to be 269
We merged (101, 110) to be 270
We merged (10, 10) to be 271
We merged (97, 114) to be 272
We merged (32, 257) to be 273
We merged (111, 110) to be 274
We merged (108, 108) to be 275
We merged (104, 97) to be 276
We merged (44, 10) to be 277
We merged (46, 271) to be 278
We merged (105, 259) to be 279
We merged (101, 115) to be 280
We merged (121, 262) to be 281
We merged (32, 115) to be 282
We merged (116, 269) to be 283
We merged (266, 260) to be 284
We merged (111, 119) to be 285
We merged (101, 97) to be 286
We merged (32, 109) to be 287
We merged (32, 119) to be 288


In [8]:
print(f"Compression ratio is {len(tokens)/len(ids):.2f}X")

Compression ratio is 1.62X


In [9]:
"""r_merges = {v:k for k,v in merges.items()}
def decode(ids):
    output = []
    for v in ids:
        if v < 257:
            output.append(chr(v))
        else:
            output += decode(list(r_merges[v]))
    return output
bruh = decode(ids)
answer = "".join(bruh)
answer"""

'r_merges = {v:k for k,v in merges.items()}\ndef decode(ids):\n    output = []\n    for v in ids:\n        if v < 257:\n            output.append(chr(v))\n        else:\n            output += decode(list(r_merges[v]))\n    return output\nbruh = decode(ids)\nanswer = "".join(bruh)\nanswer'

In [10]:
vocab = {idx : bytes([idx]) for idx in range(256)}

for (p0,p1),id in merges.items():
    vocab[id] = vocab[p0] + vocab[p1]

def decode(ids):
    byte_object = b"".join([vocab[id] for id in ids])
    text = byte_object.decode("utf-8", errors= "replace")
    return text


In [11]:
def encode(text):
    tokens = list(map(int,(text.encode("utf-8"))))
    while len(tokens) >= 2:
        stats = get_pairwise(tokens)
        pair = min(stats, key= lambda p : merges.get(p, float("inf")))
        if pair not in merges:
            break
        idxb = merges[pair]
        tokens = merge(tokens, pair, idxb)
    return tokens



In [12]:
import tiktoken 

enc1 = tiktoken.get_encoding("gpt2")

enc2 = tiktoken.get_encoding("cl100k_base")

text = "Bruh this my text let us see how the encodings deal with it"

my_tokens = encode(text)

gpt_tokens = enc1.encode(text)

print(my_tokens)
print(gpt_tokens)


[66, 114, 117, 104, 273, 279, 328, 116, 101, 120, 258, 321, 258, 117, 259, 305, 256, 104, 285, 304, 270, 99, 111, 100, 291, 259, 100, 286, 108, 288, 303, 32, 316]
[33, 622, 71, 428, 616, 2420, 1309, 514, 766, 703, 262, 2207, 375, 654, 1730, 351, 340]


In [13]:
# 1. round trip
tests = ["hello world", "Bruh this my text let us see how the encodings deal with it", "héllo 😀"]
for s in tests:
    assert decode(encode(s)) == s, f"round trip failed: {s}"
print("round trip OK")

# 2. what did it learn?
for (a, b), idx in list(merges.items())[:30]:
    print(idx, repr(vocab[idx]))

# 3. token counts vs tiktoken
gpt2 = tiktoken.get_encoding("gpt2")
gpt4 = tiktoken.get_encoding("cl100k_base")
samples = {
    "english": "Tokenization is surprisingly tricky, even for emojis.",
    "hindi": "टोकनाइज़ेशन आश्चर्यजनक रूप से मुश्किल है",
    "telugu": "టోకనైజేషన్ ఆశ్చర్యకరంగా కష్టం",
}
for name, s in samples.items():
    print(name, "| bytes:", len(s.encode("utf-8")), "| mine:", len(encode(s)),
          "| gpt2:", len(gpt2.encode(s)), "| gpt4:", len(gpt4.encode(s)))

round trip OK
256 b'e '
257 b'th'
258 b't '
259 b's '
260 b'd '
261 b', '
262 b'ou'
263 b'er'
264 b'in'
265 b'y '
266 b'an'
267 b':\n'
268 b'or'
269 b'o '
270 b'en'
271 b'\n\n'
272 b'ar'
273 b' th'
274 b'on'
275 b'll'
276 b'ha'
277 b',\n'
278 b'.\n\n'
279 b'is '
280 b'es'
281 b'you'
282 b' s'
283 b'to '
284 b'and '
285 b'ow'
english | bytes: 53 | mine: 36 | gpt2: 12 | gpt4: 10
hindi | bytes: 110 | mine: 110 | gpt2: 69 | gpt4: 46
telugu | bytes: 83 | mine: 83 | gpt2: 83 | gpt4: 54
